# Fine tunning EfficientNetV2S



In [3]:
# Intalación de dependencias.
!pip -q install kaggle

In [5]:
# Damos acceso al API token de kaggle.
!mkdir -p /root/.config/kaggle
!cp "/content/drive/MyDrive/Plant-pathology/kaggle.json" /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

In [6]:
!kaggle competitions download -c plant-pathology-2020-fgvc7

 97% 754M/779M [00:08<00:00, 186MB/s]
100% 779M/779M [00:08<00:00, 97.4MB/s]


In [8]:
# Descomprimimos la base de datos en el drive.
data_dest_folder = "/content/drive/MyDrive/Plant-pathology/data"
zip_file_path = "/content/plant-pathology-2020-fgvc7.zip"
!mkdir -p "{data_dest_folder}"
!unzip -q "{zip_file_path}" -d "{data_dest_folder}"

In [10]:
# Impotamos librerías a utilizar.
import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

import wandb
from wandb.integration.keras import WandbCallback
from wandb.integration.keras import WandbMetricsLogger

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical, image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam, AdamW, RMSprop
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

In [2]:
# Importamos los datos del drive de colab.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
os.environ["WANDB_API_KEY"] = "0df13b0c5c253813bd9eb5bfff98fdfb82fa37d0"

In [23]:
wandb.login(key=os.getenv("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: emilio-soto (emilio-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
# Ordenamos los datos.
data_path = "/content/drive/MyDrive/Plant-pathology/data"
image_dir = f"{data_path}/images"
csv_path = f"{data_path}/train.csv"

# Visualizamos.
df = pd.read_csv(csv_path)
df.head(8)

,image_id,healthy,multiple_diseases,rust,scab
0,Train_0,0,0,0,1
1,Train_1,0,1,0,0
2,Train_2,1,0,0,0
3,Train_3,0,0,1,0
4,Train_4,1,0,0,0
5,Train_5,1,0,0,0
6,Train_6,0,1,0,0
7,Train_7,0,0,0,1


In [14]:
# Convertimos a one hot encoding en una sola columna.
label_columns = ['healthy', 'multiple_diseases', 'rust', 'scab']
df['label'] = df[label_columns].idxmax(axis=1)

In [15]:
# Revisamos las imágenes por clase.
print("Conteo de imágenes por clase:")
print(df['label'].value_counts())

Conteo de imágenes por clase:
label
rust                 622
scab                 592
healthy              516
multiple_diseases     91
Name: count, dtype: int64


In [16]:
# Agregamos la extensión .jpg a las imagenes.
if not df['image_id'].str.contains('.jpg').any():
    print("Añadimos extensión .jpg a 'image_id'...")
    df['image_id'] = df['image_id'] + '.jpg'
else:
    print("'image_id' ya tiene la extensión .jpg.")

Añadimos extensión .jpg a 'image_id'...


In [17]:
# División de datos.
# Separamos el test.
main_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])

# El resto lo separamos entre entrenamiento y validación.
train_df, val_df = train_test_split(main_df, test_size=0.20, random_state=42, stratify=main_df['label'])

In [18]:
# Definimos el tamaño de las imagenes y el tamaño del batch.
IMG_SIZE = (320, 320)
BATCH_SIZE = 32
NUM_CLASSES = 4

# Creamos los generadores.
train_datagen_augmented = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest')

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Creamos los datasets.
train_dataset = train_datagen_augmented.flow_from_dataframe(
    dataframe=train_df,
    directory=image_dir,
    x_col='image_id',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42)

validation_dataset = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=image_dir,
    x_col='image_id',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False)

test_dataset = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=image_dir,
    x_col='image_id',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False)

Found 1019 validated image filenames belonging to 4 classes.
Found 255 validated image filenames belonging to 4 classes.
Found 547 validated image filenames belonging to 4 classes.


In [19]:
# Cargamos el modelo.
MODEL_PATH = "/content/drive/MyDrive/Plant-pathology/EffNetV2S_head_trained.keras"
model = load_model(MODEL_PATH)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [20]:
# Iniciamos registro en wandb.
config_fase2 = {
    "learning_rate": 1e-5,
    "architecture": "EfficientNetV2S_FULL (Fine-Tuned)",
    "batch_size": BATCH_SIZE,
    "img_size": IMG_SIZE,
    "optimizer": "Adam"}

In [24]:
wandb.init(
    project="Plant-Pathology-CNN",
    name="Entrenamiento fino",
    config=config_fase2,
    reinit=True)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [25]:
# Descongelamos la base.
model.trainable = True

In [26]:
# Compilamos.
model.compile(
    optimizer=Adam(learning_rate=config_fase2["learning_rate"]),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [27]:
# Callbacks.
callbacks_fase2 = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='final_plant_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1)]

In [28]:
history_fase2 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=40,  # 40 épocas, pero EarlyStopping decidirá
    callbacks=callbacks_fase2)
wandb.finish()

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4213 - loss: 1.2865
Epoch 1: val_accuracy improved from -inf to 0.49804, saving model to final_plant_model.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.4212 - loss: 1.2862 - val_accuracy: 0.4980 - val_loss: 1.1207
Epoch 2/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4028 - loss: 1.3034
Epoch 2: val_accuracy improved from 0.49804 to 0.50196, saving model to final_plant_model.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.4033 - loss: 1.3028 - val_accuracy: 0.5020 - val_loss: 1.1152
Epoch 3/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4021 - loss: 1.2768
Epoch 3: val_accuracy improved from 0.50196 to 0.50588, saving model to final_plant_model.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.4023 - loss: 1.2764 - val_accuracy: 0.5059 - val_loss: 1.1146
Epoch 4/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4236 - loss: 1.2575
Epoch 4: val_accuracy did not improve

epoch/accuracy,▃▃▂▂▃▃▄▆▆▄▇▅▁█▅▃▅▆█▂
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,▇█▅█▇▇▃▃▂▄▃▃▇▁▃▂▄▄▄▅
epoch/val_accuracy,▄▅▅▅▄▆▆▄▃▇▅█▅▁▅▆▂▆▄▃
epoch/val_loss,▄▃▃▃▂▂▃▂▄▂▁▁▃█▆▄▆▃▃▂
epoch/accuracy,0.40824
epoch/epoch,19
epoch/learning_rate,1e-05
epoch/loss,1.25765
epoch/val_accuracy,0.4902


In [29]:
# Cargamos el modelo otra vez.
MODEL_PATH = "final_plant_model.keras"
model = load_model(MODEL_PATH)

In [30]:
# Opción 2. Bajamos el learning rate.
config_option_2 = {
    "learning_rate": 1e-6,
    "architecture": "EfficientNetV2S_FULL (Fine-Tuned v2)",
    "batch_size": BATCH_SIZE,
    "img_size": IMG_SIZE,
    "optimizer": "Adam"}


wandb.init(
    project="Plant-Pathology-CNN",
    name="Fase3-FineTuning-LR_1e-6",
    config=config_option_2,
    reinit=True)

model.compile(
    optimizer=Adam(learning_rate=config_option_2["learning_rate"]),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

callbacks_option_2 = [
    WandbMetricsLogger(log_freq='epoch'),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1),
    ModelCheckpoint(
        filepath='final_model_option_2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1)]

In [31]:
history_option_2 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=40,
    callbacks=callbacks_option_2)

wandb.finish()

Epoch 1/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3905 - loss: 1.2904
Epoch 1: val_accuracy improved from -inf to 0.51765, saving model to final_model_option_2.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 157s 3s/step - accuracy: 0.3915 - loss: 1.2889 - val_accuracy: 0.5176 - val_loss: 1.1082
Epoch 2/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4170 - loss: 1.2499
Epoch 2: val_accuracy improved from 0.51765 to 0.52157, saving model to final_model_option_2.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.4170 - loss: 1.2498 - val_accuracy: 0.5216 - val_loss: 1.1077
Epoch 3/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4222 - loss: 1.2639
Epoch 3: val_accuracy did not improve from 0.52157
32/32 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.4224 - loss: 1.2636 - val_accuracy: 0.5059 - val_loss: 1.1080
Epoch 4/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3956 - loss: 1.2707
Epoch 4: val_accuracy did not improve from 0.52157
32/32 ━━━━━━━━━━━━━━━━━━━

epoch/accuracy,▄▃▅▃▇▂▁▇▇▃▄█
epoch/epoch,▁▂▂▃▄▄▅▅▆▇▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,▃▃▄▁▄▅█▃▁▄▃▂
epoch/val_accuracy,▇█▃▃▃▃▁▂▂▃▅▃
epoch/val_loss,▄▁▂▂▂▁▅▆█▇▇▅
epoch/accuracy,0.4475
epoch/epoch,11
epoch/learning_rate,0.0
epoch/loss,1.2375
epoch/val_accuracy,0.50588
